<a href="https://colab.research.google.com/github/tort-cam/ST554HW8/blob/main/ST554HW8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ST554 Homework 8
Author: Cameron Mullaney

This notebook features homeworks 7 and 8. \
In this assignment, I will be working with a collection of data on red and white wines from the UCI data repository. In last week's homework (HW7), I fit some models to this data to predict both alcohol content (Part I) and wine type (Part II). This week, I am building on that work, and adding to the collection of models being compared. \
<br>
In each part, I am fitting a Tree, and a Random Forest model to the data. For part I, they are regression models, and for part II, they are classification models, because of the binary nature of the response variable (red or white). These models are then tested against the each other, and the models from HW7 to determine which fits this data best.

# ST554 Homework 7
Author: Cameron Mullaney \
<br>

In this assignment, I will be working with a collection of data on red and white wines, from the UCI data repository. I will be fitting different kinds of models to the data, and the assignment is split into two parts, one predicting `alcohol`, and the other predicting `color`, or wine type (red/white).

I will be using Multiple Linear Regression, LASSO, Ridge Regression, and Elastic Net models in part I, and the analogous methods for Logistic Regression Models in part II.

### Reading in the tools we'll need

As always, we need to install a collection of math and analysis tools, as well as the package for geting the data from UCI.

In [55]:
pip install ucimlrepo

In [56]:
## Pulling in the data
from ucimlrepo import fetch_ucirepo
## The essentials
import numpy as np
import pandas as pd
import math
## Modeling & Regression
from sklearn import linear_model
import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split, cross_validate
from scipy.stats import linregress
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, ElasticNetCV, ElasticNet, RidgeCV, Ridge, LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

### Reading in our data

Now we can pull in the data using `fetch_ucirepo()`. \
We will have to bring `color` and `quality` from the total dataset (they weren't included in `features`), that way all the columns we will need will be easily accessible.

In [57]:
# fetch dataset
wine_quality = fetch_ucirepo(id=186)

In [58]:
wine = wine_quality.data['features'].copy()
wine['color'] = wine_quality.data['original']['color']
wine['quality'] = wine_quality.data['original']['quality']
wine['color'] = wine['color'].map({'red': 0, 'white': 1}) ## Red = 0, White = 1

In [59]:
wine

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,color,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,0,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,0,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,0,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,0,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,1,6
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,1,5
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,1,6
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,1,7


## Part I: Alcohol as response

### Train and test split

Here, we are splitting our data into a training and test split. This allows us to train our models on the "training" data, and still have some additional, unseen data to test our models on.

Here, the "N"s on the variables mean they are unstandardized. That will come next.

In [60]:
cols = wine.columns.drop("alcohol")

x_trainN, x_testN, y_train, y_test = train_test_split(
    wine[cols],
    wine['alcohol'],
    test_size = .2,
    random_state = 9,
    stratify = wine['color']
)

### Standardizing the training split

I am using the `StandardScaler` method to quickly standardize our data. All columns should now have mean ~0 and std ~1

In [61]:
scaler = StandardScaler()
x_train = pd.DataFrame(scaler.fit_transform(x_trainN), columns = x_trainN.columns)
x_train.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,color,quality
count,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03
mean,-1.367217e-16,1.886760e-16,6.084116e-17,3.076239e-17,1.678259e-16,1.162135e-17,9.570520e-18,3.908327e-14,3.329174e-16,-2.474663e-16,-1.032249e-16,8.886911e-17
std,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00
min,-2.655375e+00,-1.582749e+00,-2.226657e+00,-1.018009e+00,-1.302397e+00,-1.644949e+00,-1.932224e+00,-2.526757e+00,-2.960377e+00,-2.074426e+00,-1.750237e+00,-3.224840e+00
25%,-6.355791e-01,-6.720074e-01,-4.770658e-01,-7.660517e-01,-5.033603e-01,-7.518265e-01,-6.827389e-01,-7.721183e-01,-6.667797e-01,-6.714686e-01,5.713511e-01,-9.423076e-01
50%,-1.694723e-01,-3.077106e-01,-5.716387e-02,-4.930977e-01,-2.553834e-01,-8.198483e-02,3.879466e-02,6.025298e-02,-4.688844e-02,-1.370087e-01,5.713511e-01,1.989584e-01
75%,3.743190e-01,4.208830e-01,5.027054e-01,5.567254e-01,2.405704e-01,5.878569e-01,6.899347e-01,7.394679e-01,6.349919e-01,4.642587e-01,5.713511e-01,1.989584e-01
max,6.744445e+00,6.006767e+00,6.381333e+00,1.267168e+01,1.528450e+01,1.443125e+01,5.705473e+00,1.474328e+01,4.912241e+00,9.817307e+00,5.713511e-01,3.622756e+00


Looks good! Those are some small means

### MLR

The mutliple linear regression model is simplest we will use. We will take in a selection of columns from the data, and the model will produce coefficients for each column's values.

`poly` and `poly2` are settings you can give your model to tell it to create additional variables. <br>
`poly` specifies `interaction_only = True`, meaning that there will be interaction terms, but no squared terms.<br>
`poly2` includes these squared terms as well.

In [62]:
poly = PolynomialFeatures(degree = 2, interaction_only = True, include_bias = False)
poly2 = PolynomialFeatures(degree = 2, include_bias = False)

pipeline = Pipeline([
    ('interactions', poly),
    ('model', LinearRegression())
])
pipeline2 = Pipeline([
    ('interactions', poly2),
    ('model', LinearRegression())
])

In [63]:
## Full Model, includes color & quality
cv_full_mlr = cross_validate(
    LinearRegression(),
    x_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

## Medium-size model with interaction term (poly)
cv_med_int_mlr = cross_validate(
    pipeline,
    x_train[['density', 'residual_sugar', 'total_sulfur_dioxide', 'quality']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

## Small model with polynomial term (poly2)
cv_small_int_mlr = cross_validate(
    pipeline2,
    x_train[['density', 'residual_sugar']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

## Smaller model, only includes a few variables
cv_small_mlr = cross_validate(
    LinearRegression(),
    x_train[['density', 'residual_sugar']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")



In [64]:
print("MLR Models RMSE:\n" +
      "Full simple model: \t"+ str(np.sqrt(-sum(cv_full_mlr['test_score']))) + "\n" +
      "Medium interaction: \t"+ str(np.sqrt(-sum(cv_med_int_mlr['test_score']))) + "\n" +
      "Small polynomial: \t"+ str(np.sqrt(-sum(cv_small_int_mlr['test_score']))) + "\n" +
      "Small: \t\t\t"+ str(np.sqrt(-sum(cv_small_mlr['test_score']))))

MLR Models RMSE:
Full simple model: 	1.1594887389353608
Medium interaction: 	1.7118070497102211
Small polynomial: 	1.6136845111301223
Small: 			1.9468913155026373


From this print statement, we can see that to no surprise, the full simple model is best. \
Let's save that one for later use.

In [65]:
mlr_best = LinearRegression().fit(x_train, y_train)

### LASSO

Similar to the last part, here I am setting up a LASSO model, and using `LassoCV` to determine the best tuning parameter (alpha) \
I've chosen `residual_sugar`, `volatile_acidity`, `total_sulfur_dioxide`, `quality`, and `density` as my 5 predictors of `alcohol`.

In [66]:
lassomodel = LassoCV(cv=5, random_state=0).fit(x_train[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']], y_train)

Seems like .0008 is the best tuning parameter for this model!

In [67]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lassomodel.alphas_, np.mean(lassomodel.mse_path_, axis = 1))))
print(fit_info[fit_info[:,1].argsort()][0:10,].round(5))
print("alpha for lowest MSE:\t" + str(lassomodel.alpha_))

[[0.00081 0.53102]
 [0.00087 0.53103]
 [0.00093 0.53103]
 [0.001   0.53103]
 [0.00107 0.53103]
 [0.00115 0.53104]
 [0.00123 0.53104]
 [0.00132 0.53104]
 [0.00142 0.53105]
 [0.00152 0.53105]]
alpha for lowest MSE:	0.0008120305631876538


Saving this model for later!

In [68]:
lasso_best = Lasso(lassomodel.alpha_).fit(x_train[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']], y_train)

### Ridge Regression

Here I set up a ridge regression model. This is the same idea as the previous, but requires a list of potential $\alpha$ values to test.

In [69]:
ridge = RidgeCV(alphas = (range(0,100)),
                cv = 5)
ridge.fit(x_train[['fixed_acidity', 'citric_acid', 'pH', 'total_sulfur_dioxide', 'density']], y_train)
print(ridge.alpha_)

60


Seems like 60 is our best alpha value for this model! Let's save that version for later.

In [70]:
ridge_best = Ridge(alpha = ridge.alpha_)
ridge_best.fit(x_train[['fixed_acidity', 'citric_acid', 'pH', 'total_sulfur_dioxide', 'density']], y_train)

Ridge(alpha=np.int64(60))

### Elastic Net

Here I set up an Elastic Net model, again using 5 predictors. When cross validating here, I supply an list of potential `l1_ratio`s, and this method will determine which of the provided ratios is best.

In [71]:
stretch = ElasticNetCV(cv = 5,
                     l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1])
stretch.fit(x_train[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'color']], y_train)
print(stretch.alpha_, stretch.l1_ratio_)

0.010931912584990391 0.1


Looks like the model fits best at alpha of .011 and l1 ratio of .1.

In [72]:
en_best = ElasticNet(alpha = stretch.alpha_, l1_ratio = stretch.l1_ratio_)
en_best.fit(x_train[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'color']], y_train)

ElasticNet(alpha=np.float64(0.010931912584990391), l1_ratio=np.float64(0.1))

### Model Testing

Before Testing out models here, we must first standardize the test set data, based on the values from the training set. \
Here we see that our means are *pretty close* to 0, and our standard deviations are *pretty close* to 1, though not as close as they were on our training set. This makes sense given it's based on our training set.

In [73]:
x_test = pd.DataFrame(scaler.transform(x_testN), columns = x_testN.columns)
x_test.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,color,quality
count,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000
mean,-0.011056,-0.030775,0.016265,-0.005503,-0.032353,0.015786,-0.004484,-0.003991,0.029024,0.025385,-0.000117,-0.041585
std,1.035106,0.997663,1.082189,0.994829,0.811136,0.952865,0.973164,0.991985,0.983073,0.970048,1.000454,0.982259
min,-2.344637,-1.582749,-2.226657,-0.976016,-1.219738,-1.589129,-1.914626,-2.410225,-3.084355,-2.007618,-1.750237,-3.224840
25%,-0.635579,-0.732724,-0.547049,-0.766052,-0.503360,-0.751827,-0.667340,-0.832049,-0.666780,-0.671469,0.571351,-0.942308
50%,-0.169472,-0.307711,-0.057164,-0.556087,-0.255383,-0.081985,0.038795,0.083559,-0.046888,-0.137009,0.571351,0.198958
75%,0.374319,0.360167,0.572689,0.556725,0.268123,0.643677,0.742730,0.779422,0.634992,0.531066,0.571351,0.198958
max,6.045285,7.524671,9.390630,4.325590,8.368702,3.769605,2.537765,2.993529,3.362513,7.078200,0.571351,3.622756


First, I am testing based on RMSE, taking the `mean_squared_error`, and using `np.sqrt()` to return this value. \
Second, I am testing based on MAE, taking the `mean_absolute_error`. \
<br>
Through both metrics, the full MLR model is best by a wide margin, with the LASSO and Ridge models fitting about equal to each other.

In [74]:
MLR = mlr_best.predict(x_test)
LASSO = lasso_best.predict(x_test[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])
RIDGE = ridge_best.predict(x_test[['fixed_acidity', 'citric_acid', 'pH', 'total_sulfur_dioxide', 'density']])
EN = en_best.predict(x_test[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'color']])

print(" Model \t RMSE \t\t\t MAE \n",

      "MLR \t", np.sqrt(mean_squared_error(y_test,MLR)),"\t",
        mean_absolute_error(y_test,MLR) , "\n" ,

      "LASSO \t", np.sqrt(mean_squared_error(y_test,LASSO)),"\t",
        mean_absolute_error(y_test,LASSO), "\n" ,

      "RIDGE \t", np.sqrt(mean_squared_error(y_test,RIDGE)),"\t",
        mean_absolute_error(y_test,RIDGE), "\n" ,

      "EN \t", np.sqrt(mean_squared_error(y_test,EN)),"\t",
        mean_absolute_error(y_test,EN))

 Model 	 RMSE 			 MAE 
 MLR 	 0.465313441450464 	 0.3532999763427778 
 LASSO 	 0.7207570271254976 	 0.5659253062120058 
 RIDGE 	 0.7342792074412888 	 0.5788971766994361 
 EN 	 1.0370458878929019 	 0.8464594645168972


## Part II: Wine Type as Response

This section is similar to the work done above, but now using **color** (a value of either 1 or 0) as our response variable.

### Train and test split

Here, the "2"s differentiate these variables from those above. The "N"s on the variables mean they are unstandardized. That will come next.

In [75]:
cols2 = wine.columns.drop("color")

x_train2N, x_test2N, y_train2, y_test2 = train_test_split(
    wine[cols2],
    wine['color'],
    test_size = .2,
    random_state = 9
)

### Standardizing the training split

Great! Our new test split features alcohol and not color! Standardizing, same as before.

In [76]:
scaler = StandardScaler()
x_train2 = pd.DataFrame(scaler.fit_transform(x_train2N), columns = x_train2N.columns)
x_train2.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
count,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03
mean,3.212960e-17,1.189479e-16,-4.443456e-17,4.101651e-17,8.476746e-17,-3.691486e-17,-4.101651e-18,-2.863431e-14,-1.935979e-15,-2.707090e-16,-8.681829e-17,-1.985883e-16
std,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00
min,-2.556906e+00,-1.575624e+00,-2.196872e+00,-9.944079e-01,-1.346069e+00,-1.657909e+00,-1.933141e+00,-2.514192e+00,-3.082083e+00,-2.119514e+00,-2.084752e+00,-3.239155e+00
25%,-6.342340e-01,-6.735697e-01,-4.710288e-01,-7.662740e-01,-5.189089e-01,-7.662304e-01,-6.839694e-01,-7.876485e-01,-6.780120e-01,-6.999271e-01,-8.317708e-01,-9.378715e-01
50%,-1.727926e-01,-3.127480e-01,-5.682632e-02,-5.174007e-01,-2.622041e-01,-9.747110e-02,3.738338e-02,7.232839e-02,-6.158354e-02,-1.591323e-01,-1.635141e-01,2.127702e-01
75%,3.655556e-01,4.088954e-01,4.954436e-01,5.610505e-01,2.512055e-01,6.270181e-01,7.059542e-01,7.444942e-01,6.164878e-01,4.492619e-01,6.718067e-01,2.127702e-01
max,6.671921e+00,7.444918e+00,6.294278e+00,1.250697e+01,1.579611e+01,1.439231e+01,5.702642e+00,1.457661e+01,4.869844e+00,9.913172e+00,3.678962e+00,3.664695e+00


### Logistic Regression Model

These `PolynomialFeatures` are the same as in part 1, the pipelines have been renamed for clarity, and as to not overwrite part 1's pipelines. \
Here the models are built using `LogisticRegression()`, which is applicable for a binary reponse variable.

In [77]:
poly = PolynomialFeatures(degree = 2, interaction_only = True, include_bias = False)
poly2 = PolynomialFeatures(degree = 2, include_bias = False)

no_inter = Pipeline([
    ('interactions', poly),
    ('model', LogisticRegression())
])
inter = Pipeline([
    ('interactions', poly2),
    ('model', LogisticRegression())
])

In [78]:
## Full Model, includes alcohol & quality
cv_full_logreg_color = cross_validate(
    LogisticRegression(),
    x_train2,
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")

## Medium-size model with interaction term
cv_med_int_logreg_color = cross_validate(
    no_inter,
    x_train2[['density', 'residual_sugar', 'total_sulfur_dioxide', 'quality']],
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")

## Small model with polynomial term - Chosen based on early plot
cv_small_int_logreg_color = cross_validate(
    inter,
    x_train2[['density', 'residual_sugar']],
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")

## Smaller model, only includes a few variables - Chosen based on early plot
cv_small_logreg_color = cross_validate(
    LogisticRegression(),
    x_train2[['density', 'residual_sugar']],
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")



In [79]:
print(" Log Reg Model \t\t Mean Log Loss\t\t St. Dev of Log Loss \n\n" ,

      "Full simple model: \t", -cv_full_logreg_color['test_score'].mean(), "\t",
                                cv_full_logreg_color['test_score'].std(), "\n" ,

      "Medium interaction: \t", -cv_med_int_logreg_color['test_score'].mean(), "\t",
                                 cv_med_int_logreg_color['test_score'].std(), "\n" ,

      "Small polynomial: \t", -cv_small_int_logreg_color['test_score'].mean(), "\t",
                               cv_small_int_logreg_color['test_score'].std(), "\n" ,

      "Small: \t\t" , -cv_small_logreg_color['test_score'].mean(), "\t",
                       cv_small_logreg_color['test_score'].std())

 Log Reg Model 		 Mean Log Loss		 St. Dev of Log Loss 

 Full simple model: 	 0.03925872016100645 	 0.019821020372276354 
 Medium interaction: 	 0.05881398115455644 	 0.010544113654825337 
 Small polynomial: 	 0.1452853793834913 	 0.011290852267735607 
 Small: 		 0.15242335746715496 	 0.012134215307452024


From this print statement, we can see that the full simple model is best. However, we do see the full model has a much higher standard deviation.\
We will save the full model for later use.

In [80]:
logreg_basic_best = LogisticRegression().fit(x_train2, y_train2) ## Keeps all columns

### LASSO

Similar to the last part, here I am setting up a LASSO model, and using `LassoCV` to determine the best tuning parameter (alpha) \
I've chosen `residual_sugar`, `volatile_acidity`, `total_sulfur_dioxide`, `quality`, and `density` as my 5 predictors of `alcohol`.

In part II, it's simpler here to design these models using `LogisticRegressionCV()`, as opposed to `cross_validate()` with different styles. `penalty = "l1"` specifies we want a LASSO model.

In [81]:
lassologreg = LogisticRegressionCV(
    penalty = "l1",
    Cs = np.linspace(.1,10,20),
    cv = 5,
    solver = "liblinear",
    scoring = "neg_log_loss"
)

In [82]:
lassologreg.fit(x_train2[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']], y_train2)

LogisticRegressionCV(Cs=array([ 0.1       ,  0.62105263,  1.14210526,  1.66315789,  2.18421053,
        2.70526316,  3.22631579,  3.74736842,  4.26842105,  4.78947368,
        5.31052632,  5.83157895,  6.35263158,  6.87368421,  7.39473684,
        7.91578947,  8.43684211,  8.95789474,  9.47894737, 10.        ]),
                     cv=5, penalty='l1', scoring='neg_log_loss',
                     solver='liblinear')

Seems like 0.62 is the best C parameter for this model! ($\alpha$ = 1.61)

In [83]:
np.set_printoptions(suppress = True)
fit_info2 = np.array(list(zip(lassologreg.Cs_, -np.mean(lassologreg.scores_[1], axis=0))))
print(fit_info2[fit_info2[:,1].argsort()][0:10,].round(5))
print("C for lowest Log Loss:\t" , lassologreg.C_)

[[0.62105 0.0494 ]
 [1.14211 0.04976]
 [1.66316 0.04993]
 [2.18421 0.05003]
 [2.70526 0.0501 ]
 [3.22632 0.05014]
 [3.74737 0.0502 ]
 [4.26842 0.05021]
 [4.78947 0.05023]
 [5.31053 0.05025]]
C for lowest Log Loss:	 [0.62105263]


Saving this model for later!

In [84]:
lasso_logreg_best = LogisticRegression(
    penalty = "l1",
    C = float(lassologreg.C_[0]),
    solver = "liblinear")\
    .fit(x_train2[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']], y_train2)

### Ridge Regression

Here I set up a ridge regression model. `penalty = "l2"` specifies a Ridge Regression model.

In [85]:
Ridgelogreg = LogisticRegressionCV(
    penalty = "l2",
    Cs = np.linspace(.1,10,20),
    cv = 5,
    scoring = "neg_log_loss"
)

In [86]:
Ridgelogreg.fit(x_train2, y_train2)

LogisticRegressionCV(Cs=array([ 0.1       ,  0.62105263,  1.14210526,  1.66315789,  2.18421053,
        2.70526316,  3.22631579,  3.74736842,  4.26842105,  4.78947368,
        5.31052632,  5.83157895,  6.35263158,  6.87368421,  7.39473684,
        7.91578947,  8.43684211,  8.95789474,  9.47894737, 10.        ]),
                     cv=5, scoring='neg_log_loss')

Looks like for the Ridge Log Reg Model, C should be 7.394? ($\lambda$ = .135)

Even all the way out at 10 digits, the log loss at C = 7.915 and 7.395 are equal. 7.916 is listed first in the list, but the model has chosen 7.394 as its C value.

In [87]:
np.set_printoptions(suppress = True)
RLR_fit_info = np.array(list(zip(Ridgelogreg.Cs_, -np.mean(Ridgelogreg.scores_[1], axis=0))))
print(RLR_fit_info[RLR_fit_info[:,1].argsort()][0:10,].round(20))
print("C for lowest Log Loss:\t" , Ridgelogreg.C_)

[[7.91578947 0.03911076]
 [7.39473684 0.03911076]
 [6.87368421 0.0391132 ]
 [4.26842105 0.03913383]
 [3.22631579 0.03913411]
 [5.31052632 0.03914673]
 [4.78947368 0.0391561 ]
 [3.74736842 0.03915637]
 [0.62105263 0.03916962]
 [5.83157895 0.03919017]]
C for lowest Log Loss:	 [7.39473684]


Let's save that model for later.

In [88]:
RLR_best = LogisticRegression(
    penalty = "l2",
    C = float(Ridgelogreg.C_[0]),
    solver = "liblinear") \
    .fit(x_train2, y_train2)

### Elastic Net

Here I set up an Elastic Net model, again using 5 predictors. When cross validating here, I supply an list of potential `l1_ratio`s, and this method will determine which of the provided ratios is best. `penalty = "elasticnet"` specifies the approach. \
Note - These did not converge as quickly, so `max_iter` had to be increased from its default value.

In [89]:
ENlogreg = LogisticRegressionCV(
    penalty = "elasticnet",
    l1_ratios = [.05, 0.1, 0.5, 0.9, 0.96,0.99, 1],
    Cs = 10,
    cv = 5,
    solver = "saga",
    scoring = "neg_log_loss",
    max_iter = 1000
)

In [90]:
ENlogreg.fit(x_train2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']], y_train2)
print(ENlogreg.C_, ENlogreg.l1_ratio_)

[0.35938137] [0.05]


Looks like the model fits best at C of .36 and l1 ratio of .05. Let's save it!

In [91]:
best_EN_logreg = LogisticRegression(
    penalty = "elasticnet",
    l1_ratio = .05,
    C = float(ENlogreg.C_[0]),
    solver = "saga",
    max_iter = 1000) \
    .fit(x_train2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']], y_train2)

### Model Testing

Same as in part I, we must first standardize the test set data. \
Here we see that our means are *pretty close* to 0, and our standard deviations are *pretty close* to 1, though not as close as they were on our training set. This makes sense given it's based on our training set.

In [92]:
x_test2 = pd.DataFrame(scaler.transform(x_test2N), columns = x_test2N.columns)
x_test2.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
count,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000
mean,-0.036017,-0.070327,0.013854,-0.053411,-0.022657,-0.062297,-0.011487,-0.072211,-0.045888,-0.076765,-0.016487,0.018931
std,0.984616,0.947235,1.015745,0.930589,0.996062,0.942970,0.971843,0.936452,0.953978,1.026961,0.981231,1.023657
min,-2.633813,-1.575624,-2.196872,-1.015147,-1.203455,-1.657909,-1.915547,-2.477948,-2.404012,-1.916716,-2.084752,-3.239155
25%,-0.634234,-0.711155,-0.540063,-0.766274,-0.518909,-0.766230,-0.617992,-0.805771,-0.739655,-0.767526,-0.831771,-0.937871
50%,-0.249700,-0.312748,-0.056826,-0.517401,-0.262204,-0.153201,0.019789,-0.046289,-0.123226,-0.226732,-0.163514,0.212770
75%,0.288649,0.288621,0.564477,0.457353,0.194160,0.515558,0.635578,0.645646,0.554845,0.314063,0.671807,0.212770
max,6.364293,5.400262,9.262729,3.547531,15.824631,5.113278,3.045952,2.770877,3.575344,9.575175,2.927173,3.664695


First, I am testing based on Log loss (closer to 0 is better)\
Second, I am testing based on Accuracy (closer to 1 is better)\
<br>
Through both metrics, the full MLR model is best, with the LASSO and Ridge models fitting about equal to each other, just slightly worse than the simple Log Reg model.

In [93]:
BASIClog = logreg_basic_best.predict(x_test2)
LASSOlog = lasso_logreg_best.predict(x_test2[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])
RIDGElog = RLR_best.predict(x_test2)
ENlog = best_EN_logreg.predict(x_test2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']])
print("Fit for different Logistic Regression models \n")
print(" Type \t Log Loss \t\t Accuracy \n", "-" * 50, "\n",

      "BASIC \t", log_loss(y_test2,BASIClog),"\t",
        accuracy_score(y_test2,BASIClog) , "\n" ,

      "LASSO \t", log_loss(y_test2,LASSOlog),"\t",
        accuracy_score(y_test2,LASSOlog), "\n" ,

      "RIDGE \t", log_loss(y_test2,RIDGElog),"\t",
        accuracy_score(y_test2,RIDGElog), "\n" ,

      "EN \t", log_loss(y_test2,ENlog),"\t",
        accuracy_score(y_test2,ENlog))

Fit for different Logistic Regression models 

 Type 	 Log Loss 		 Accuracy 
 -------------------------------------------------- 
 BASIC 	 0.2495329850015805 	 0.9930769230769231 
 LASSO 	 0.36043653389117175 	 0.99 
 RIDGE 	 0.30498475944637615 	 0.9915384615384616 
 EN 	 2.8557663839069742 	 0.9207692307692308


#New Homework 8 Work
Here we go! New work, as described at the top of the notebook.

First, I am importing additional tools I will need.

In [94]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV

## Part I - Regression

These models will use alcohol content as the response variable. Because this is continuous, a regression model is used.

### Regression Tree

Here I'm fitting our data to a regression tree model, and using `GridSearchCV` to determine what my best parameters will be. `GridSearchCV` will return the optimal parameters from the list of provided parameters below.

In [95]:
parameters = {'max_depth':range(2,10),
              'min_samples_leaf':[2,5,10,25,50]}

In [96]:
#LASSOlog = lasso_logreg_best.predict(x_test2[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])

tree_cv = GridSearchCV(DecisionTreeRegressor(),
                       parameters,
                       cv = 5,
                       scoring = 'neg_mean_squared_error')
tree_cv.fit(x_train[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']], y_train)

print("best parameters", tree_cv.best_params_, "\n", "best RMSE", np.sqrt(-tree_cv.best_score_))

best parameters {'max_depth': 9, 'min_samples_leaf': 5} 
 best RMSE 0.6049839981420104


In [97]:
## Saving best fit for later testing

tree_best = tree_cv.best_estimator_

Seems like `max_depth` should be 9, and `min_samples_leaf` should be 5. We end up with a RMSE of .604

### Random Forest

Here I'm fitting a random forest model to our data, and using cross_validate to determine the best value for `max_features`.

With 12 predictors, this range will cover both `sqrt(predictors)` and predictors/3

In [98]:
forestparams = {'max_features': range(1,5)}

In [99]:
#forest_cv = cross_validate(RandomForestRegressor(forestparams),
forest_cv = GridSearchCV(RandomForestRegressor(),
                         forestparams,
                         cv = 5,
                         scoring = 'neg_mean_squared_error')
forest_cv.fit(x_train, y_train)
print("best parameters", forest_cv.best_params_, "\n", "best RMSE", np.sqrt(-forest_cv.best_score_))

best parameters {'max_features': 4} 
 best RMSE 0.44435364595583826


In [100]:
## Saving best fit for later testing

forest_best = forest_cv.best_estimator_

Seems like `max_features` of 4 is our best bet with the full dataset of variables.

### Model Testing

Here, the models from HW7 will be compared with the two above models with **root mean squared error**, and **mean absolute error** as the metrics.

In [101]:
## Copied from HW7

MLR = mlr_best.predict(x_test)
LASSO = lasso_best.predict(x_test[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])
RIDGE = ridge_best.predict(x_test[['fixed_acidity', 'citric_acid', 'pH', 'total_sulfur_dioxide', 'density']])
EN = en_best.predict(x_test[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'color']])

## New Models

TREE = tree_best.predict(x_test[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])
FOREST = forest_best.predict(x_test)

print(" Model \t RMSE \t\t\t MAE \n",

      "MLR \t", np.sqrt(mean_squared_error(y_test,MLR)),"\t",
        mean_absolute_error(y_test,MLR) , "\n" ,

      "LASSO \t", np.sqrt(mean_squared_error(y_test,LASSO)),"\t",
        mean_absolute_error(y_test,LASSO), "\n" ,

      "RIDGE \t", np.sqrt(mean_squared_error(y_test,RIDGE)),"\t",
        mean_absolute_error(y_test,RIDGE), "\n" ,

      "EN \t", np.sqrt(mean_squared_error(y_test,EN)),"\t",
        mean_absolute_error(y_test,EN), "\n",

      "TREE \t", np.sqrt(mean_squared_error(y_test,TREE)),"\t",
        mean_absolute_error(y_test,TREE) , "\n" ,

      "FOREST ", np.sqrt(mean_squared_error(y_test,FOREST)),"\t",
        mean_absolute_error(y_test,FOREST))


 Model 	 RMSE 			 MAE 
 MLR 	 0.465313441450464 	 0.3532999763427778 
 LASSO 	 0.7207570271254976 	 0.5659253062120058 
 RIDGE 	 0.7342792074412888 	 0.5788971766994361 
 EN 	 1.0370458878929019 	 0.8464594645168972 
 TREE 	 0.5991839096657608 	 0.4465306694043275 
 FOREST  0.4362809095164023 	 0.300759635373212


From this table, it seems like our Random forest model is best! It narrowly surpasses the full MLR model from last week.

## Part II - Classification

Here, I am doing similar tasks to in Part I, but using Wine Type as the response. Because wine type is binary, these are *classification models* as opposed to regression as used earlier.

### Classification Tree

Here I'm fitting our data to a classification tree model, and using `GridSearchCV` to determine what my best parameters will be.

In [102]:
ct_parameters = {'max_depth':range(2,10),
              'min_samples_leaf':[2,5,10,25,50,75,100]}

In [103]:
ctree_cv = GridSearchCV(DecisionTreeClassifier(random_state = 9),
                       ct_parameters,
                       cv = 5,
                       scoring = 'neg_log_loss')
ctree_cv.fit(x_train2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']], y_train2)

print("best parameters", ctree_cv.best_params_, "\nbest Log Loss", -ctree_cv.best_score_)

best parameters {'max_depth': 5, 'min_samples_leaf': 50} 
best Log Loss 0.1477596593615287


Seems like `max_depth` should be 5, and `min_samples_leaf` should be 50. We end up with a log loss of .148 - pretty good!

In [104]:
## Saving best fit for later testing

ctree_best = ctree_cv.best_estimator_

### Random Classification Forest

Here I'm fitting a random forest model to our data, and using cross_validate to determine the best value for `max_features`. Again, this is a classification model.

With 12 predictors, this range will cover both `sqrt(predictors)` and predictors/3

In [105]:
cforestparams = {'max_features': range(2,6)}

In [106]:
cforest_cv = GridSearchCV(RandomForestClassifier(random_state = 9),
                         cforestparams,
                         cv = 5,
                         scoring = 'neg_log_loss')

cforest_cv.fit(x_train2, y_train2)
print("best parameters", cforest_cv.best_params_, "\n", "best log loss", -cforest_cv.best_score_)

best parameters {'max_features': 5} 
 best log loss 0.034762291131428


In [107]:
## Saving best fit for later testing

cforest_best = cforest_cv.best_estimator_

Seems like `max_features` of 5 is our best bet with the full dataset of variables.

### Model Testing

Here, the models from HW7 will be compared with the two above models with **Log loss** and **Accuracy** as the metrics.

In [108]:
## Copied from HW7

BASIClog = logreg_basic_best.predict(x_test2)
LASSOlog = lasso_logreg_best.predict(x_test2[['residual_sugar', 'volatile_acidity', 'total_sulfur_dioxide', 'quality', 'density']])
RIDGElog = RLR_best.predict(x_test2)
ENlog = best_EN_logreg.predict(x_test2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']])

## New Models

c_TREE = ctree_best.predict(x_test2[['fixed_acidity', 'chlorides', 'residual_sugar', 'sulphates', 'quality']])
c_FOREST = cforest_best.predict(x_test2)

## Print!

print("Fit for different Logistic & Classification models \n")
print(" Type \t Log Loss \t\t Accuracy \n", "-" * 50, "\n",

      "BASIC \t", log_loss(y_test2,BASIClog),"\t",
        accuracy_score(y_test2,BASIClog) , "\n" ,

      "LASSO \t", log_loss(y_test2,LASSOlog),"\t",
        accuracy_score(y_test2,LASSOlog), "\n" ,

      "RIDGE \t", log_loss(y_test2,RIDGElog),"\t",
        accuracy_score(y_test2,RIDGElog), "\n" ,

      "EN \t", log_loss(y_test2,ENlog),"\t",
        accuracy_score(y_test2,ENlog), "\n",

      "TREE \t", log_loss(y_test2,c_TREE),"\t",
        accuracy_score(y_test2,c_TREE), "\n" ,

      "FOREST ", log_loss(y_test2,c_FOREST),"\t",
        accuracy_score(y_test2,c_FOREST))



Fit for different Logistic & Classification models 

 Type 	 Log Loss 		 Accuracy 
 -------------------------------------------------- 
 BASIC 	 0.2495329850015805 	 0.9930769230769231 
 LASSO 	 0.36043653389117175 	 0.99 
 RIDGE 	 0.30498475944637615 	 0.9915384615384616 
 EN 	 2.8557663839069742 	 0.9207692307692308 
 TREE 	 1.6912791205662667 	 0.953076923076923 
 FOREST  0.1940812105567849 	 0.9946153846153846


Once again, it seems like the Forest of classication trees is the best option once again! It leads in both log loss and accuracy.